# 07: Sentiment Analysis (Category-Level, Internal Feature)

Adds a news-sentiment feature to the recommendation engine, per your
supervisor's request. Requirements confirmed with her:

- **Internal, not user-facing** -- computed once ahead of time, shown
  automatically alongside predictions (no new form input)
- **Based on news/articles** -- not general market-return data
- **Category-level** -- matches how `predicted_min_cagr` already works,
  avoids the same sparsity problem that motivated category-based ML
  features in `05` (138 funds is too many to get reliable, fund-specific
  news coverage for each one individually)
- **Data source**: NewsAPI (free tier) -- her guidance was "anywhere it's
  available"

## What this notebook does
1. Loads the final checkpoint from `06` (`risk_return_df_138funds_filtered_min7yr.csv`)
2. Pulls recent news headlines per category (7 searches, one per category)
3. Runs headlines through FinBERT (a RoBERTa-family model fine-tuned for
   financial text) to get a sentiment score per headline
4. Aggregates to one sentiment score per category
5. Merges the score onto `risk_return_df_filtered` as a new column
6. Saves a **new** checkpoint (`risk_return_df_138funds_with_sentiment.csv`)
   -- does not overwrite the existing filtered checkpoint, so the
   pre-sentiment version stays available if this needs to be rolled back

## Before running this notebook
- Sign up for a free API key at newsapi.org
- `pip install requests transformers torch --break-system-packages`

## Setup

In [4]:
import torch 
import transformers
print(torch.__version__)
print(transformers.__version__)

2.13.0+cpu
5.14.1


In [5]:
import pandas as pd
import requests

# Paste your NewsAPI key here
NEWSAPI_KEY = "34c04d2576af4328b84e405d9cf14d39"



## Load the existing checkpoint

Starting point -- the same filtered, validated dataset the web app currently runs on.

In [6]:
risk_return_df_filtered = pd.read_csv('../Data/external/risk_return_df_138funds_filtered_min7yr.csv')

print(risk_return_df_filtered.shape)
risk_return_df_filtered[['fund', 'window_years', 'mean', 'min']].head()

(744, 9)


,fund,window_years,mean,min
0,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,1,0.165303,-0.201841
1,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,2,0.158986,-0.052699
2,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,3,0.154762,0.005373
3,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,4,0.155859,0.052588
4,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,5,0.157720,0.026134


## Category list

The 7 target categories used throughout the project. Search queries below
are written in plain English per category, tuned to return relevant
financial news rather than unrelated results.

In [7]:
CATEGORY_SEARCH_QUERIES = {
    'Equity Scheme - Large Cap Fund': 'Indian large cap mutual funds',
    'Equity Scheme - Mid Cap Fund': 'Indian mid cap mutual funds',
    'Equity Scheme - Small Cap Fund': 'Indian small cap mutual funds',
    'Debt Scheme - Corporate Bond Fund': 'Indian corporate bond funds',
    'Debt Scheme - Short Duration Fund': 'Indian short duration debt funds',
    'Hybrid Scheme - Aggressive Hybrid Fund': 'Indian hybrid mutual funds',
    'Other Scheme - Index Funds': 'Nifty Sensex index funds India'
}

print(len(CATEGORY_SEARCH_QUERIES), "categories")

7 categories


## Step 1: Test a single NewsAPI call

Run this first, on its own, before looping over all 7 categories --
confirms the API key works and the query returns sensible results.

In [8]:
def get_headlines(query, api_key, page_size=20):
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "apiKey": api_key,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": page_size
    }
    response = requests.get(url, params=params)
    return response.json()

# Test with one category first
test_result = get_headlines("Indian small cap mutual funds", NEWSAPI_KEY)
print(test_result.get('totalResults', 'ERROR -- check response below'))
print(test_result)

28
{'status': 'ok', 'totalResults': 28, 'articles': [{'source': {'id': None, 'name': 'BusinessLine'}, 'author': 'Team BL, KS Badri Narayanan', 'title': 'Sensex today | Stock Market Highlights: Sensex falls 715 points, Nifty slips below 24,000; media, realty drag, FMCG and auto resilient', 'description': 'Sensex, Nifty updates on 22nd July 2026: Indian equity benchmarks posted their biggest single-day decline in two weeks on Wednesday as escalating tensions in the Middle East drove Brent crude above $95 a barrel, raising concerns over inflation and economic gr…', 'url': 'https://www.thehindubusinessline.com/markets/sensex-nifty50-today-stock-market-highlights-22nd-july-2026/article71249266.ece', 'urlToImage': 'https://bl-i.thgim.com/public/incoming/rpky4p/article71252052.ece/alternates/LANDSCAPE_1200/Stockmarket.jpg', 'publishedAt': '2026-07-22T10:54:13Z', 'content': '<li></li>\r\nJuly 22, 2026 16:21The live blog is now closed.\r\n<li></li>\r\nJuly 22, 2026 15:58DSP Mutual Fund launches

**Check before continuing**: did `test_result` return real articles, or an
error message (e.g. invalid API key, rate limit)? If `totalResults` is 0 or
missing, inspect the full `test_result` dict printed above -- NewsAPI
returns a clear `message` field explaining what went wrong.

## Notes

(placeholder -- remaining steps: fetch all 7 categories, load FinBERT, score headlines, aggregate, merge, save checkpoint. Continue building once Step 1 is confirmed working.)

In [10]:
import time
all_articles = {}

for category , query in CATEGORY_SEARCH_QUERIES.items():
    results = get_headlines(query, NEWSAPI_KEY, page_size=20)
    articles = results.get('articles', [])
    descriptions = [a['description'] for a in articles if a.get('description')]
    all_articles[category] = descriptions
    print(f" {category} : {len(descriptions)}")
    time.sleep(1)


 Equity Scheme - Large Cap Fund : 20
 Equity Scheme - Mid Cap Fund : 20
 Equity Scheme - Small Cap Fund : 20
 Debt Scheme - Corporate Bond Fund : 19
 Debt Scheme - Short Duration Fund : 9
 Hybrid Scheme - Aggressive Hybrid Fund : 19
 Other Scheme - Index Funds : 20


we took descriptions instead of total data because total data had too much noise which was not usefull and could cloud the judgement of the model. so we use descriptions instead as they will be of the matching category only

In [12]:
from transformers import pipeline
sentiment_analyzer = pipeline('sentiment-analysis' , model='ProsusAI/finbert')

c:\Users\verma\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\verma\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18253.48it/s]


pipeline(...) is a Hugging Face transformers helper function — it's a shortcut that bundles together three things you'd otherwise have to set up manually: downloading the model weights, downloading the matching tokenizer (the thing that converts text into numbers the model understands), and wiring them together into one callable object.

After importing the lib checking that it works or not

In [13]:
print(sentiment_analyzer("The market showed strong gains today"))

[{'label': 'positive', 'score': 0.9500779509544373}]


Now step 3: scoring every description we fetched earlier and aggregating into one sentiment number per category.

In [14]:
category_sentiment ={}
for cat, desc in all_articles.items():
    if len(desc) ==0:
        category_sentiment[cat] =0
        continue
    results = sentiment_analyzer(desc, truncation = True)

    signed_scores = []
    for r in results:
        if r['label'] == 'positive':
            signed_scores.append(r['score'])
        elif r['label'] == 'negative':
            signed_scores.append(-r['score'])
        else:
            signed_scores.append(0)

    avg_sentiment = sum(signed_scores)/len(signed_scores)
    category_sentiment[cat] =avg_sentiment
    print(f"{cat} : {avg_sentiment:.4f} ({len(desc)} articles )")

Equity Scheme - Large Cap Fund : 0.1073 (20 articles )
Equity Scheme - Mid Cap Fund : 0.1005 (20 articles )
Equity Scheme - Small Cap Fund : 0.1235 (20 articles )
Debt Scheme - Corporate Bond Fund : -0.1555 (19 articles )
Debt Scheme - Short Duration Fund : 0.0818 (9 articles )
Hybrid Scheme - Aggressive Hybrid Fund : 0.1690 (19 articles )
Other Scheme - Index Funds : -0.2603 (20 articles )


Now creating a table to use and merge the data

In [15]:
sentiment_df = pd.DataFrame([{'Scheme_Category' : cat, 'sentiment_score': score} for cat, score in category_sentiment.items()])
sentiment_df

,Scheme_Category,sentiment_score
0,Equity Scheme - Large Cap Fund,0.107255
1,Equity Scheme - Mid Cap Fund,0.100490
2,Equity Scheme - Small Cap Fund,0.123495
3,Debt Scheme - Corporate Bond Fund,-0.155541
4,Debt Scheme - Short Duration Fund,0.081827
5,Hybrid Scheme - Aggressive Hybrid Fund,0.168965
6,Other Scheme - Index Funds,-0.260310


MERGING THE SENTIMENT SCORE WITH CAT CODE INTO THE FINAL FILE

In [16]:
final_selection = pd.read_csv('../Data/external/final_selection_140funds.csv')
category_lookup = final_selection[['Scheme_Code', 'Scheme_Category']].drop_duplicates()

category_lookup['Scheme_Category'] = category_lookup['Scheme_Category'].replace(
    'Income/Debt Oriented Schemes - Corporate Bond Fund',
    'Debt Scheme - Corporate Bond Fund'
)

risk_return_df_with_sentiment = risk_return_df_filtered.merge(
    category_lookup, on='Scheme_Code', how='left'
)
risk_return_df_with_sentiment = risk_return_df_with_sentiment.merge(
    sentiment_df, on='Scheme_Category', how='left'
)

print(risk_return_df_with_sentiment.shape)
print(risk_return_df_with_sentiment['sentiment_score'].isna().sum(), "NaN (expect 0)")
risk_return_df_with_sentiment[['fund', 'Scheme_Category', 'mean', 'min', 'sentiment_score']].head()

(744, 11)
0 NaN (expect 0)


,fund,Scheme_Category,mean,min,sentiment_score
0,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,Equity Scheme - Large Cap Fund,0.165303,-0.201841,0.107255
1,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,Equity Scheme - Large Cap Fund,0.158986,-0.052699,0.107255
2,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,Equity Scheme - Large Cap Fund,0.154762,0.005373,0.107255
3,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,Equity Scheme - Large Cap Fund,0.155859,0.052588,0.107255
4,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,Equity Scheme - Large Cap Fund,0.157720,0.026134,0.107255


In [17]:
risk_return_df_with_sentiment.to_csv('../Data/external/risk_return_df_138funds_with_sentiment.csv', index=False)